# Adding a Dataset with an Agentic Workflow

Notebook 2 added a dataset by hand: you wrote the harmonizer and dataset
classes yourself. This notebook does the same job a different way. You hand the
mechanical scaffolding to a coding agent (Claude Code) and stay in the
reviewer's seat.

The repo ships a skill file, [docs/add_dataset_skill.md](../docs/add_dataset_skill.md),
that encodes the exact recipe for a new dataset (config, harmonizer, dataset,
exports, integrity check). An agent that reads it can scaffold a dataset from a
plain-language description, so your job shrinks to three things: point at the
data, write the prompt, and verify the result.

We reuse the same toy Montgomery County chest X-ray set as notebook 2, so you
can compare the hand-written and agent-driven paths side by side.

## How it works

1. **You describe the data.** Where the images live, what the metadata CSV
   holds, and which columns map to patient / study / image / labels.
2. **The agent does the scaffolding.** Guided by the skill, it writes
   `configs/datasets/my_montgomery.json`, the harmonizer and dataset folders,
   wires the package exports, and runs the integrity check, pausing for you to
   review each edit.
3. **You verify.** Load the dataset by name and run the same check CI uses.

The agent runs in your terminal or IDE (Claude Code), opened in the repo root.
This notebook holds the two pieces you own: the data description you feed it
(below) and the verification you run afterward.

In [ ]:
import pandas as pd

# Point at the toy data. base_image_dir is the folder holding every PNG;
# the metadata CSV ships with the tutorial (relative to this folder).
INDEX_CSV = "data/montgomery_toy.csv"
MONTGOMERY_DIR = "/mnt/NAS4/datasets/external/Montgomery-CXR/MontgomerySet/CXR_png/"

raw = pd.read_csv(INDEX_CSV)
print("rows:", len(raw), "| columns:", list(raw.columns))
raw.head()

## The prompt

Open Claude Code in the repo root and describe the dataset. You can do this two
ways; both name the skill and give it the data layout and column mapping you
just inspected.

**Free-text prompt.** Fastest to write, good when the mapping is simple:

> Use the add-dataset skill in `docs/add_dataset_skill.md` to add a new dataset
> with registry key `my_montgomery`.
>
> The data is the toy Montgomery County TB chest X-ray set: a folder of PNGs at
> `/mnt/NAS4/datasets/external/Montgomery-CXR/MontgomerySet/CXR_png/` plus a
> metadata CSV at `data/montgomery_toy.csv` with one row per image and columns
> `pid` (patient id), `fname` (image filename), and `tb` (binary TB label).
> There is one study per patient. Only the classification output is needed (no
> masks, bounding boxes, or reports).
>
> Follow the skill end to end (config, harmonizer, dataset, exports, integrity
> check) and show me each file before you write it.

**Structured prompt.** The skill's first step is to capture every decision in
`configs/datasets/<key>.json`. Handing the agent that JSON up front removes
ambiguity: it pins the paths, the column mapping, and the label handling before
any code is written, and it doubles as a spec you can review at a glance. Give a
compact spec and let the agent expand it into the full config:

The `derivation` field is where plain language does the work. When a canonical
column is a direct copy of a raw column, set `derivation` to `null` (as with
`pid` -> `patient_id` below). When it needs computing, describe the transform in
words and let the agent write the code to match, for example:

- `"mirror patient_id"` (reuse another canonical column)
- `"strip the leading files/ prefix from the path"`
- `"take the second underscore-separated field of fname"`
- `"concatenate study and image id with a slash"`

You describe the intent; the agent turns each `derivation` string into the
matching `_build_*` hook in the harmonizer.

> Use the add-dataset skill in `docs/add_dataset_skill.md` to add a dataset from
> this spec, then follow the recipe end to end (config, harmonizer, dataset,
> exports, integrity check) and show me each file before you write it:
>
> ```json
> {
>   "dataset_name": "My Montgomery (toy)",
>   "registry_key": "my_montgomery",
>   "modality": "2D",
>   "image_format": "PNG",
>   "base_image_dir": "/mnt/NAS4/datasets/external/Montgomery-CXR/MontgomerySet/CXR_png/",
>   "primary_csv": {"path": "data/montgomery_toy.csv"},
>   "patient_id": {"column": "pid", "derivation": null},
>   "study_id": {"column": "pid", "derivation": "mirror patient_id"},
>   "image_path": {"column": "fname", "derivation": null},
>   "labels": {"columns": ["tb"], "separate_csv": null},
>   "masks": {"supported": false},
>   "bboxes": {"supported": false},
>   "reports": {"supported": false}
> }
> ```

Either way, the agent reads the skill, mirrors an existing CSV-driven dataset,
and proposes the edits. Review them as they arrive: the recipe is deterministic,
but you are the one who confirms the mapping and paths are right before
accepting.

## Verify the result

Once the agent has written the files, run the two cells below. The first imports
the new dataset submodule directly, which registers it in this session. If you
ran these cells once before the files existed, restart the kernel (Kernel >
Restart) so the import is fresh. If `my_montgomery` still does not resolve, the
agent has not finished writing the dataset module.

In [ ]:
import matplotlib.pyplot as plt

# Import the new dataset submodule directly. This runs its @register_dataset
# decorator right away, so the dataset is live in this session without waiting
# on the top-level package exports (those load on the next kernel start).
import radharmony.dataset.my_montgomery.my_montgomery  # noqa: F401
from radharmony.registry import resolve_dataset, list_datasets

# Paths again, so this cell stands alone after a kernel restart.
INDEX_CSV = "data/montgomery_toy.csv"
MONTGOMERY_DIR = "/mnt/NAS4/datasets/external/Montgomery-CXR/MontgomerySet/CXR_png/"

print("registered:", "my_montgomery" in list_datasets())

MyMontgomeryDataset = resolve_dataset("my_montgomery")
ds = MyMontgomeryDataset(
    base_image_dir=MONTGOMERY_DIR,
    csv_path=INDEX_CSV,
    output_cls=True,
    cache_dir=None,
)
print("harmonized rows:", len(ds.get_harmonized_df()))

# One figure per sample; build, then display each.
built = ds.get_datasets()
for fig in ds.visualize_samples(built, n=3):
    display(fig)
    plt.close(fig)

In [ ]:
from radharmony.integrity import check_dataset, print_report, subsample_dataset

# The same check CI runs: harmonize, load a subsample, validate shape/dtype/range/labels.
check_ds = MyMontgomeryDataset(
    base_image_dir=MONTGOMERY_DIR,
    csv_path=INDEX_CSV,
    output_cls=True,
    cache_dir=None,
)
subsample_dataset(check_ds, max_samples=30)  # fast pass over a subset
report = check_dataset(check_ds, num_workers=0)  # 0 workers is notebook-safe
print_report(report)

## Recap

The agent did the mechanical scaffolding; you owned the parts that need
judgment: describing the data, reviewing each edit, and verifying the result.
On disk you now have the same four pieces notebook 2 built inline, but written
into the package so the dataset survives a kernel restart and loads by name
anywhere:

- `configs/datasets/my_montgomery.json`
- `radharmony/harmonizer/my_montgomery/`
- `radharmony/dataset/my_montgomery/`
- exports appended to the two package `__init__.py` files

To finish a real contribution, ask the agent to add a wiki page under
`docs/wiki/datasets/` and an inventory row in `docs/wiki/datasets.md`, then open
the pull request. The full checklist lives in
[docs/add_dataset_skill.md](../docs/add_dataset_skill.md).